Quiz 56 — Call Centre Shift Optimisation  [SOLVED]
Difficulty: Hard

In [ ]:
import numpy as np

np.random.seed(190)
calls       = np.random.poisson(lam=12, size=(4,5,3,5))
shift_costs = np.array([45, 50, 60])
shift_names = ["Morning","Afternoon","Evening"]

# ── 1. Total calls per shift-slot (4,5,3) ─────────────────────────────────────
shift_totals = calls.sum(axis=3)            # sum over agents → (4,5,3)
print("Shift totals shape:", shift_totals.shape)

# ── 2. Base cost (4,5,3) ──────────────────────────────────────────────────────
# shift_costs (3,) → broadcast over (4,5,3): 5 agents per shift × cost per shift
base_cost = 5 * shift_costs                 # per shift: 5 agents each
# broadcast: (3,) over (4,5,3) aligns on last axis
base_grid = np.ones_like(shift_totals, dtype=float) * base_cost  # (4,5,3)

# ── 3. Overtime ───────────────────────────────────────────────────────────────
overtime_calls = np.maximum(shift_totals - 80, 0)   # clip negatives to 0
overtime_cost  = overtime_calls * 0.50

total_cost_grid = base_grid + overtime_cost          # (4,5,3)

# ── 4. Weekly cost ────────────────────────────────────────────────────────────
weekly_cost = total_cost_grid.sum(axis=(1,2))        # sum days + shifts → (4,)
print("\nWeekly costs:")
for wk, cost in enumerate(weekly_cost):
    print(f"  Week {wk+1}: £{cost:,.2f}")

# ── 5. Most overtime incidents per shift type ─────────────────────────────────
ot_incidents = (overtime_calls > 0)                  # boolean (4,5,3)
ot_per_shift = ot_incidents.sum(axis=(0,1))          # sum over weeks and days → (3,)
most_ot_shift = np.argmax(ot_per_shift)
print("\nOvertime incidents per shift type:")
for name, cnt in zip(shift_names, ot_per_shift):
    print(f"  {name:11s}: {cnt} incidents")
print(f"Most overtime: {shift_names[most_ot_shift]}")

# ── 6. 90th percentile staffing buffer ───────────────────────────────────────
print("\n90th pct calls by shift (staffing buffer):")
for i, name in enumerate(shift_names):
    # shift_totals[:,:,i] extracts all weeks/days for that shift type
    p90 = np.percentile(shift_totals[:,:,i], 90)
    print(f"  {name:11s}: {p90:.0f} calls → recommend staff for this volume")

# ── WHY ───────────────────────────────────────────────────────────────────────
# np.maximum(arr, 0) is the vectorised equivalent of max(x, 0) per element —
# clips negative differences to zero.
# Staffing at the 90th percentile means you'll handle 90% of days without
# overtime — a practical capacity planning approach.